# 3.6 能不能偷看答案？

# 第 3 章：Attention，自己決定看哪裡

前置：向量、矩陣乘法和 softmax。先用單 head；每個小節都畫得出矩陣。完整模型仍留到第4章。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：不能偷看明天的答案**

對角線可看；右上角未來位置禁止。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/causal.svg")))

**先想一想：**改最後一格的 value，第一格輸出應該變嗎？

第 t 個位置是在猜下一個 token；它可以看目前與過去，不能看未來的答案。對角線可見，所以不是嚴格下三角。

**把直覺寫成數學：**allowed[i,j]=(j≤i)；被禁止的 score 設為 −∞，softmax 後權重0。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.attention import manual_attention

q, k, v = [torch.randn(1, 1, 4, 2) for _ in range(3)]
allowed = torch.ones(4, 4, dtype=torch.bool).tril()[None, None]
out, weights = manual_attention(q, k, v, allowed)
changed = v.clone()
changed[:, :, 3] += 100
other, _ = manual_attention(q, k, changed, allowed)
print(weights[0, 0])
assert torch.allclose(out[:, :, :3], other[:, :, :3])

**如何讀結果：**反例檢查「不偷看」比只是看 loss 正常更重要。

**只改一件事：**只移除 tril mask，確認反例開始失敗。
